In [ ]:
"""
import and config
"""

import os
from pathlib import Path
import torch
from sdp.reader.reader_factory import ReaderFactory
from sdp.processor.processor_factory import ProcessorFactory
from sdp.processor.processors.HwProcessor import HwProcessor
from sdp.processor.processors.BfeeProcessor import BfeeProcessor
from sdp.processor.processors.WiproxProcessor import WiproxProcessor
from sdp.utils.csi_process_utils import batch_preprocess
from sklearn.model_selection import train_test_split
from sdp.dataset.CSIDateset import CSIDataset
from sdp.dataset.CSIDateset import csi_collate_fn
from torch.utils.data import DataLoader

# 设置设备
device = torch.device("cuda")
print("使用设备:", device)

# params
PROJECT_ROOT = Path.cwd()
os.chdir(PROJECT_ROOT)

folder_path = PROJECT_ROOT / "data/widar_data"
task_type = 'classification'
sub_type = 'Gesture Recognition'
final_fs = 1000
num_samples = 100000
batch_size = 32

In [ ]:
"""
模型
"""


import torch
import torch.nn as nn

class AdaptiveCSIModel(nn.Module):
    """自适应CSI深度学习模型，优化版"""

    def __init__(self, task_type, num_classes=None, output_dim=None):
        super(AdaptiveCSIModel, self).__init__()
        self.task_type = task_type

        self.conv1 = nn.Conv3d(2, 16, kernel_size=(3, 3, 3), padding=1, groups=2)
        self.bn1 = nn.BatchNorm3d(16)
        self.pool1 = nn.MaxPool3d((1, 2, 1))

        self.conv2 = nn.Conv3d(16, 32, kernel_size=(3, 3, 3), padding=1, groups=16)
        self.bn2 = nn.BatchNorm3d(32)
        self.pool2 = nn.MaxPool3d((2, 2, 1))

        self.conv3 = nn.Conv3d(32, 64, kernel_size=(3, 3, 3), padding=1, groups=32)
        self.bn3 = nn.BatchNorm3d(64)
        self.pool3 = nn.AdaptiveAvgPool3d((None, 1, 1))

        self.time_conv = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.lstm = nn.LSTM(64, 128, batch_first=True, bidirectional=True)

        if task_type == 'classification':
            if num_classes is None:
                raise ValueError("num_classes must be provided for classification tasks")
            self.fc = nn.Linear(256, num_classes)
        elif task_type == 'regression':
            if output_dim is None:
                output_dim = 1
            self.fc = nn.Linear(256, output_dim)
        else:
            raise ValueError("task_type must be either 'classification' or 'regression'")

    def forward(self, x):
        B, T, Freq, Nt, Nr, C = x.shape
        x = x.permute(0, 5, 1, 2, 3, 4)  # [B, 2, T, Freq, Nt, Nr]
        x = x.reshape(B, 2, T, Freq, Nt*Nr)  # [B, 2, T, Freq, P]

        x = torch.nn.functional.relu(self.bn1(self.conv1(x)))
        x = self.pool1(x)

        x = torch.nn.functional.relu(self.bn2(self.conv2(x)))
        x = self.pool2(x)

        x = torch.nn.functional.relu(self.bn3(self.conv3(x)))
        x = self.pool3(x)
        x = x.squeeze(-1).squeeze(-1)  # [B, 64, T]

        x = torch.nn.functional.relu(self.time_conv(x))
        x = x.permute(0, 2, 1)  # [B, T, 64]

        self.lstm.flatten_parameters()
        x, _ = self.lstm(x)
        x = x[:, -1, :]

        return self.fc(x)

def train_model(model, train_loader, val_loader, task_type, num_epochs=10, lr=0.001):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    if task_type == 'classification':
        criterion = nn.CrossEntropyLoss()
    else:
        # regression task
        criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=2)

    best_val_loss = float('inf')

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)

        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)

                if task_type == 'classification':
                    _, predicted = torch.max(outputs.data, 1)
                    correct += (predicted == labels).sum().item()
                    total += labels.size(0)

        val_loss /= len(val_loader.dataset)
        scheduler.step(val_loss)

        if task_type == 'classification':
            val_acc = correct / total if total > 0 else 0
            print(
                f'Epoch {epoch + 1}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}')
        else:
            print(f'Epoch {epoch + 1}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'best_model.pth')

    print('Training complete')
    return model

In [ ]:
"""
执行位置
"""

# 遍历目录下的文件 调整逻辑获取需要的文件
folder_path = Path(folder_path)
if not folder_path.exists() or not folder_path.is_dir():
    raise ValueError(f"无效的文件夹路径: {folder_path}")
files = [f for f in folder_path.rglob("*") if f.is_file() and "truth" not in f.name]
if not files:
    raise IOError(f"文件夹 {folder_path} 中没有文件")

# 使用第一个文件确定主格式
sample_file = files[0]
reader = ReaderFactory.create_reader(str(sample_file))
sample_frame = reader.read_file(sample_file).frames[0]
processor = ProcessorFactory.get_processor(sample_frame)

print(f"检测到主文件格式: {type(reader).__name__}\n")

print(f"开始处理 {len(files)} 个文件...\n")

csi_data_list = []
# 处理所有文件
for file_path in files:
    try:
        csi_data = reader.read_file(str(file_path))
        csi_data_list.append(csi_data)

        print(f"√ 已处理: {file_path.name}\n")

    except Exception as e:
        print(f"× 处理失败 {file_path.name}: {str(e)}\n")

print(f"处理完成! 共处理 {len(files)} 个文件\n")

# 情况处理
global res
if type(processor) == HwProcessor:
    res = list(processor.process(csi_data_list, folder_path=folder_path))
elif type(processor) == BfeeProcessor:
    res = processor.process(csi_data_list, folder_path=folder_path, sub_type=sub_type, final_fs=final_fs)
elif type(processor) == WiproxProcessor:
    res = processor.process(csi_data_list, folder_path=folder_path, num_samples=num_samples)

# 处理csi
print("Preprocessing CSI data...\n")
csi_data_list = res[0]
labels = res[1]
processed_csi_data = batch_preprocess(
    csi_data_list,
    denoise=True,
    normalize=True,
    phase_correction=True
)
print("process dataset success\n")
train_data, val_data, train_labels, val_labels = train_test_split(
    processed_csi_data, labels, test_size=0.2, random_state=42
)
print(f"Train samples: {len(train_data)}, Validation samples: {len(val_data)}\n")
train_dataset = CSIDataset(train_data, train_labels, task_type)
val_dataset = CSIDataset(val_data, val_labels, task_type)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=csi_collate_fn,
    num_workers=4,  # 使用多进程加载
    pin_memory=False  # 加速GPU传输
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=csi_collate_fn,
    num_workers=2,
    pin_memory=False
)

num_classes = len(set(labels))
if task_type == 'classification':
    model = AdaptiveCSIModel(task_type, num_classes=num_classes)
else:
    model = None  # for more task_type in the future
print("Model architecture:")
print(model)

print("\nStarting training...")
trained_model = train_model(
    model,
    train_loader,
    val_loader,
    task_type
)

torch.save(trained_model.state_dict(), "final_csi_model.pth")
print("Final model saved to final_csi_model.pth\n")